In [55]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_chroma import Chroma 


In [58]:
# embeddings
embeddings = OllamaEmbeddings(model="granite-embedding")


# load vector DB
vector_db = Chroma(
    persist_directory="../model_vector_db_v1", embedding_function=embeddings
)


vector_retriever = vector_db.as_retriever(
    search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20}
)

prompt = ChatPromptTemplate.from_template("""
You are a legal assistant helping analyze a Model Law.

Answer the question using ONLY the provided context.

Context:
{context}

Question:
{question}

Answer clearly and cite sections if possible.
"""
                                          )

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [41]:
llm = OllamaLLM(
    model="deepseek-r1:8b"
)

rag_chain = (
    {
        "context": vector_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

question = "Give me list of TOC in the model law 565?"

response = rag_chain.invoke(question)

print(response)

Based on the provided context, here is the Table of Contents (TOC) for Model Law 565:

1.  Section 1. Group Life Insurance Definition
2.  Section 2. Limits of Group Life Insurance
3.  Section 3. Notice of Compensation
4.  Section 4. Dependent Group Life Insurance
5.  Section 5. Group Life Insurance Standard Provisions
6.  Section 6. Supplementary Bill Relating to Conversion Privileges


In [59]:
docs = vector_db.similarity_search("table of contents ", k=5)

for d in docs:
    print("PATH:", d.metadata)
    print(d.page_content[:500])
    print("------")

PATH: {'type': 'subsection', 'id': 'model-law-565::Section 3 > B', 'section': 'Section 3.', 'path': 'Section 3 > B', 'document': 'model-law-565'}
B. The notice shall be distributed:
------
PATH: {'document': 'model-law-565', 'type': 'subsection', 'id': 'model-law-565::Section 5 > G', 'path': 'Section 5 > G', 'section': 'Section 5.'}
G. The policy shall contain a provision that the insurer will issue to the policyholder for delivery to each person insured a certificate setting forth a statement as to the insurance protection to which he or she is entitled, to whom the insurance benefits are payable, a statement as to any dependent’s coverage included in the certificate, and the rights and conditions set forth in Subsections H, I, J and K following.
------
PATH: {'id': 'model-law-565::Section 1 > B > (5)', 'document': 'model-law-565', 'section': 'Section 1.', 'type': 'clause', 'path': 'Section 1 > B > (5)'}
(5) The insurance may be payable to the creditor or any successor to the right, t

In [60]:
import json
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

# load chunks
with open("../docs/model_law/model-law-565_chunks.json") as f:
    chunks = json.load(f)

documents = [Document(page_content=c["text"],metadata=c["metadata"]) for c in chunks]

print("Loaded documents:", len(documents))

# create BM25 retriever
bm25 = BM25Retriever.from_documents(documents)

bm25.k = 5

Loaded documents: 76


In [44]:
query = "table of contents"

docs = bm25.invoke(query)

for d in docs:
    print("PATH:", d.metadata)
    print(d.page_content[:300])
    print("------")

PATH: {'path': 'TOC', 'section': 'TOC', 'document': 'model-law-565', 'type': 'toc'}
This is the table of contents of Model Law 565.

Sections included in this model law are:

Section 1. Group Life Insurance Definition
Section 2. Limits of Group Life Insurance
Section 3. Notice of Compensation
Section 4. Dependent Group Life Insurance
Section 5. Group Life Insurance Standard Provisi
------
PATH: {'path': 'Section 1. > E. > (1)', 'section': 'Section 1.', 'document': 'model-law-565', 'type': 'clause'}
(1) The policy may insure members of the association or associations, employees thereof or employees of members, or one or more of the preceding or all of any class or classes thereof for the benefit of persons other than the employee’s employer.
------
PATH: {'path': 'Section 5. > H. > (2)', 'section': 'Section 5.', 'document': 'model-law-565', 'type': 'clause'}
(2) The individual policy shall be in an amount not in excess of the amount of life insurance that ceases because of termination, 

In [61]:
def rrf_fusion(bm25_docs, vector_docs, k=60, top_k=5):
    scores = {}
    doc_lookup = {}

    # BM25
    for rank, doc in enumerate(bm25_docs):
        key = doc.metadata["id"]

        doc_lookup[key] = doc
        scores[key] = scores.get(key, 0) + 1 / (k + rank + 1)

    # Vector
    for rank, doc in enumerate(vector_docs):
        key = doc.metadata["id"]

        doc_lookup[key] = doc
        scores[key] = scores.get(key, 0) + 1 / (k + rank + 1)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [doc_lookup[key] for key, _ in ranked[:top_k]]

In [62]:

bm25_docs = bm25.invoke(query)
vector_docs = vector_retriever.invoke(query)

results = rrf_fusion(bm25_docs, vector_docs)

In [63]:
results

[Document(metadata={'id': 'model-law-565::TOC', 'path': 'TOC', 'section': 'TOC', 'document': 'model-law-565', 'type': 'toc'}, page_content='This is the table of contents of Model Law 565.\n\nSections included in this model law are:\n\nSection 1. Group Life Insurance Definition\nSection 2. Limits of Group Life Insurance\nSection 3. Notice of Compensation\nSection 4. Dependent Group Life Insurance\nSection 5. Group Life Insurance Standard Provisions\nSection 6. Supplementary Bill Relating to Conversion Privileges'),
 Document(id='78f52953-c5af-4d03-95bd-988301a29d12', metadata={'id': 'model-law-565::Section 3 > B', 'document': 'model-law-565', 'type': 'subsection', 'section': 'Section 3.', 'path': 'Section 3 > B'}, page_content='B. The notice shall be distributed:'),
 Document(metadata={'id': 'model-law-565::Section 1 > E > (1)', 'path': 'Section 1 > E > (1)', 'section': 'Section 1.', 'document': 'model-law-565', 'type': 'clause'}, page_content='(1) The policy may insure members of the a

In [64]:
import boto3

boto3.Session().get_credentials()